# Student Academic Success Prediction: ML vs DL Comparison (Task 1: Academic Success Prediction)

**Project:** Team 11 - AI Introduction Final Project
**Team:** Park Jae-woo (Leader), Yum Ji-hun, Oh Hyung-woo
**Task 1:** Student Academic Success Prediction (학업 성공 예측)
**Goal:** Predicting if a student will drop out (y = 1: Graduate, y = 0: Non-Graduate (Graduate/Enrolled))


## Section 1: Setup & Data Loading

In [ ]:
# ===== Section 1.1: Import Libraries =====

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Deep Learning

# Class Imbalance
from imblearn.over_sampling import SMOTE
from scipy.stats import mode

print('[OK] All libraries loaded successfully!')

# ===== [복사한 코드: Assignment3.ipynb Setup & Dataset Load] =====
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ===== Section 1.2: Load Dataset =====
import os
import pandas as pd

# Google Colab 환경 감지 및 파일 업로드
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('[INFO] Colab environment detected. Please upload dataset.csv using the file selector below.\n')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('[ERROR] No file was uploaded. Please upload dataset.csv to proceed.')
    filename = list(uploaded.keys())[0]
else:
    filename = 'dataset.csv'
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"[ERROR] '{filename}' not found in the current directory. "
            "Please make sure the dataset file is placed in the same folder as this notebook."
        )
    print(f'[INFO] Local environment detected. Using local file: {filename}')

# Load CSV data
df = pd.read_csv(filename)

print(f'[INFO] Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'[INFO] Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\n[INFO] Data types:\n{df.dtypes}')
print(f'\n[INFO] Missing values:\n{df.isnull().sum()}')
print(f'\n[INFO] Target variable distribution:')
print(df['Target'].value_counts())

## Section 2: Data Preprocessing

In [ ]:
# ===== Section 2.1: Categorical Encoding & Scaling =====

# Separate features and target
X = df.drop('Target', axis=1)
y = df['Target']
# ===== [Task 2: 학생 학업 성공 예측용 Binary Target 설정] =====
# 학업성공(Graduate)는 1, 나머지(Graduate, Enrolled)는 0으로 이진분류 라벨을 생성합니다.
y_encoded = (y == 'Graduate').astype(int)
print('[OK] Binary Target (Graduate: 1, Non-Graduate: 0) defined successfully!')
print('Target Distribution:')
print(y_encoded.value_counts())

# Encode target variable

# Encode categorical features
X_encoded = X.copy()
categorical_features = X.select_dtypes(include=['object']).columns
for col in categorical_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
print(f'[OK] Categorical features encoded: {len(categorical_features)} features')

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print(f'[OK] StandardScaler applied - Mean: {X_scaled.mean().mean():.6f}, Std: {X_scaled.std().mean():.6f}')


In [ ]:
# ===== Section 2.2: Train-Test Split & SMOTE =====

# Train-test split (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
print(f'[OK] Train-test split: {X_train.shape[0]} train, {X_test.shape[0]} test')

# Apply SMOTE to balance classes
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f'[OK] SMOTE applied: {X_train.shape[0]} -> {X_train_balanced.shape[0]} samples')
print(f'[OK] Balanced class distribution:')
unique, counts = np.unique(y_train_balanced, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {("Non-Graduate" if u == 0 else "Graduate")}: {c} ({c/len(y_train_balanced)*100:.1f}%)')

## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ===== Section 3.1: Class Distribution Visualization =====

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original distribution
y.value_counts().plot(kind='bar', ax=axes[0, 0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_title('Original Class Distribution', fontsize=12, fontweight='bold')

# Percentage
y.value_counts(normalize=True).plot(kind='bar', ax=axes[0, 1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 1].set_title('Original Class Distribution (%)', fontsize=12, fontweight='bold')

# Before SMOTE
unique_before, counts_before = np.unique(y_train, return_counts=True)
axes[1, 0].bar(['Non-Graduate', 'Graduate'], counts_before, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 0].set_title('Before SMOTE', fontsize=12, fontweight='bold')

# After SMOTE
unique_after, counts_after = np.unique(y_train_balanced, return_counts=True)
axes[1, 1].bar(['Non-Graduate', 'Graduate'], counts_after, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 1].set_title('After SMOTE', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('[OK] Class distribution analysis complete')

In [ ]:
# ===== Section 3.2: Feature Distribution & Correlation =====

# Top 12 features distribution
important_features = X.columns[:12].tolist()
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, feature in enumerate(important_features):
    X_scaled[feature].hist(bins=30, ax=axes[idx], color='#45B7D1', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print('[OK] Feature distribution analysis complete')

In [ ]:
# ===== Section 3.3: PCA Dimension Reduction & Visualization =====
# 과제 3 (Assignment3.ipynb)의 주성분 분석(PCA) 기법을 이식하여 고차원 학생 데이터를 2차원으로 축소하고 
# 데이터의 전체적인 클래스 분포 패턴을 시각화합니다.
from sklearn.decomposition import PCA

print('[PROCESSING] PCA Dimension Reduction (2D Projection)...')

# PCA 모델 생성 및 2차원 변환 수행 (정규화된 balanced 훈련 세트 적용)
pca_2d = PCA(n_components=2, random_state=42)
X_train_pca = pca_2d.fit_transform(X_train_balanced)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_train_pca[:, 0], 
    X_train_pca[:, 1], 
    c=y_train_balanced, 
    cmap='coolwarm', 
    alpha=0.6, 
    edgecolors='w', 
    linewidths=0.5
)
plt.title('2D PCA Projection of Balanced Student Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)

# 클래스 레이블 범례 추가
labels = ['Non-Dropout', 'Dropout'] if 'dropout' in filename.lower() else ['Non-Graduate', 'Graduate']
handles, _ = scatter.legend_elements()
plt.legend(handles, labels, loc="upper right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Retention 분산 비율 출력 (과제 3 스타일 분석용 정보 제공)
explained_variance = pca_2d.explained_variance_ratio_
print(f"[OK] PCA Retained Variance Ratio (2 components): {sum(explained_variance)*100:.2f}%")
print(f"  - PC1 explained variance: {explained_variance[0]*100:.2f}%")
print(f"  - PC2 explained variance: {explained_variance[1]*100:.2f}%")

## Section 4: Model Development

Train and evaluate 3 Machine Learning models (Logistic Regression, SVM, Random Forest) and 1 PyTorch Deep Learning MLP model.


In [ ]:
# ===== Section 4.1: Logistic Regression =====

print('[TRAINING] Logistic Regression...')

lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10], 'penalty': ['l2']}

grid_lr = GridSearchCV(lr_model, param_grid_lr, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_lr.fit(X_train_balanced, y_train_balanced)

y_pred_lr = grid_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f'[OK] Best params: {grid_lr.best_params_}')
print(f'[OK] Accuracy: {accuracy_lr:.4f}, Precision: {precision_lr:.4f}, Recall: {recall_lr:.4f}, F1: {f1_lr:.4f}')

In [ ]:
# ===== Section 4.2: SVM =====

print('[TRAINING] SVM (RBF Kernel)...')

svm_model = SVC(kernel='rbf', random_state=42, probability=True)
param_grid_svm = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.1]}

grid_svm = GridSearchCV(svm_model, param_grid_svm, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_svm.fit(X_train_balanced, y_train_balanced)

y_pred_svm = grid_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f'[OK] Best params: {grid_svm.best_params_}')
print(f'[OK] Accuracy: {accuracy_svm:.4f}, Precision: {precision_svm:.4f}, Recall: {recall_svm:.4f}, F1: {f1_svm:.4f}')

In [ ]:
# ===== Section 4.3: Random Forest =====

print('[TRAINING] Random Forest...')

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
param_grid_rf = {'max_depth': [10, 15, 20], 'min_samples_split': [5, 10]}

grid_rf = GridSearchCV(rf_model, param_grid_rf, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_rf.fit(X_train_balanced, y_train_balanced)

y_pred_rf = grid_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f'[OK] Best params: {grid_rf.best_params_}')
print(f'[OK] Accuracy: {accuracy_rf:.4f}, Precision: {precision_rf:.4f}, Recall: {recall_rf:.4f}, F1: {f1_rf:.4f}')

In [ ]:
# ===== [복사한 코드: Assignment3.ipynb 2. Neural Network] =====
# PyTorch를 사용하여 과제3의 스타일 및 아키텍처와 완벽히 호환되도록 구성합니다.
import torch.nn as nn
import torch.optim as optim

class PyTorchMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # nn.Sequential을 활용한 Multi-Layer Perceptron 설계 (ReLU 사용)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 2)  # 이진 분류를 위한 2차원 출력
        )
        
    def forward(self, x):
        return self.net(x)

print('[OK] PyTorch MLP model defined successfully!')

# ===== [복사한 코드: Assignment2.ipynb 4. Implementing Evaluation Metrics] =====
# 과제 2에서 직접 작성한 ClassificationMetrics 클래스를 그대로 이식합니다.
import numpy as np
class ClassificationMetrics:
    def __init__(self, y_true, y_pred, y_prob=None):
        self.y_true = np.array(y_true)
        self.y_pred = np.array(y_pred)
        self.y_prob = np.array(y_prob) if y_prob is not None else None
        self.classes = np.unique(y_true)

    def confusion_matrix(self):
        # Calculate confusion matrix
        n_classes = len(self.classes)
        cm = np.zeros((n_classes, n_classes), dtype=int)
        for i, true_class in enumerate(self.classes):
            for j, pred_class in enumerate(self.classes):
                cm[i, j] = np.sum((self.y_true == true_class) & (self.y_pred == pred_class))
        return cm

    def accuracy(self):
        # Calculate accuracy
        return np.mean(self.y_true == self.y_pred)

    def precision_recall_f1(self):
        # Calculate precision, recall, and F1 for each class
        cm = self.confusion_matrix()
        n_classes = len(self.classes)
        precision = np.zeros(n_classes)
        recall = np.zeros(n_classes)
        f1 = np.zeros(n_classes)
        
        for i in range(n_classes):
            tp = cm[i, i]
            fp = np.sum(cm[:, i]) - tp
            fn = np.sum(cm[i, :]) - tp
            
            precision[i] = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall[i] = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i]) if (precision[i] + recall[i]) > 0 else 0
            
        return precision, recall, f1

    def macro_metrics(self):
        # Calculate macro-averaged metrics
        precision, recall, f1 = self.precision_recall_f1()
        return {
            'macro_precision': np.mean(precision),
            'macro_recall': np.mean(recall),
            'macro_f1': np.mean(f1)
        }

print('[OK] Custom ClassificationMetrics class imported successfully!')

# ===== [복사한 코드: Assignment3.ipynb 2. Neural Network (Train Loop)] =====
# PyTorch DataLoader를 생성하고 모델을 50 Epoch 동안 학습합니다.
print('[TRAINING] PyTorch MLP Neural Network...')

# 1. PyTorch Tensor로 데이터 변환
X_train_t = torch.tensor(X_train_balanced.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train_balanced.values, dtype=torch.long)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

# 2. Dataset 및 DataLoader 생성
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# 3. 모델 객체 선언 (과제 3 스타일)
input_dim = X_train_t.shape[1]
mlp_model = PyTorchMLP(input_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# 4. 학습 루프 수행 (과제 3 train 함수 스타일 - Loss & Acc History 추가 수집)
epochs = 50
loss_hist = []
acc_hist = []

for epoch in range(epochs):
    mlp_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = mlp_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    # 에폭마다 평가 및 히스토리 누적
    mlp_model.eval()
    with torch.no_grad():
        logits_test = mlp_model(X_test_t)
        preds = logits_test.argmax(dim=1)
        acc = (preds == y_test_t).float().mean().item()
    
    loss_hist.append(avg_loss)
    acc_hist.append(acc)
    
    # 에폭 진행 상황을 과제3 양식으로 출력
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f} - acc: {acc:.3f}')

# ===== [복사한 코드: Assignment3.ipynb 2. MLP Training History Visualization] =====
# 과제 3의 Loss와 Accuracy 추이 그래프 시각화를 이식하여 학습 상태를 시각적으로 분석합니다.
plt.figure(figsize=(10, 5))
plt.plot(loss_hist, label='Training Loss', color='#FF6B6B', linewidth=2)
plt.plot(acc_hist, label='Test Accuracy', color='#45B7D1', linewidth=2)
plt.title("PyTorch MLP Training History", fontsize=12, fontweight='bold')
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Value", fontsize=10)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 5. 최종 평가 수행 및 예측 결과 도출
mlp_model.eval()
with torch.no_grad():
    logits_test = mlp_model(X_test_t)
    y_pred_mlp = logits_test.argmax(dim=1).numpy()
    # 확률 계산 (ClassificationMetrics에 사용할 확률 값 추출)
    y_prob_mlp = torch.softmax(logits_test, dim=1).numpy()

print('[OK] PyTorch MLP Training completed successfully!')

## Section 5: Results Analysis

In [ ]:
# ===== [복사한 코드: Assignment2.ipynb Naive Bayes & KNN Verification] =====
# 직접 구현한 ClassificationMetrics 클래스를 기반으로 모든 모델을 평가합니다.

print('[EVALUATION] Using Custom ClassificationMetrics...')

# 각 모델의 ClassificationMetrics 생성
metrics_lr = ClassificationMetrics(y_test, y_pred_lr, grid_lr.predict_proba(X_test))
metrics_svm = ClassificationMetrics(y_test, y_pred_svm, grid_svm.predict_proba(X_test))
metrics_rf = ClassificationMetrics(y_test, y_pred_rf, grid_rf.predict_proba(X_test))
metrics_mlp = ClassificationMetrics(y_test, y_pred_mlp, y_prob_mlp)

# 평가 결과 저장용 리스트 생성
models_list = ['Logistic Regression', 'SVM', 'Random Forest', 'MLP (PyTorch)']
metrics_objs = [metrics_lr, metrics_svm, metrics_rf, metrics_mlp]

accuracies = []
precisions_pos = []
recalls_pos = []
f1s_pos = []

for m in metrics_objs:
    accuracies.append(m.accuracy())
    prec, rec, f1_val = m.precision_recall_f1()
    # 이진 분류이므로 양성 클래스 (index 1)의 지표를 개별 추출하여 분석
    precisions_pos.append(prec[1])
    recalls_pos.append(rec[1])
    f1s_pos.append(f1_val[1])

# 성능 비교 데이터프레임 구축
comparison_df = pd.DataFrame({
    'Model': models_list,
    'Accuracy': accuracies,
    'Precision (Class 1)': precisions_pos,
    'Recall (Class 1)': recalls_pos,
    'F1-Score (Class 1)': f1s_pos
})

print('\nModel Performance Comparison (Target Class Evaluation):')
print(comparison_df.to_string(index=False))

best_idx = comparison_df['F1-Score (Class 1)'].idxmax()
print(f'\n[BEST] Best Model: {comparison_df.loc[best_idx, "Model"]}')
print(f'[BEST] F1-Score: {comparison_df.loc[best_idx, "F1-Score (Class 1)"]:.4f}')

In [ ]:
# ===== [복사한 코드: Assignment2.ipynb 4. Confusion Matrix Comparison] =====
# 우리가 직접 구현한 ClassificationMetrics의 confusion_matrix 함수를 시각화합니다.
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

models_info = [
    ('Logistic Regression', metrics_lr),
    ('SVM', metrics_svm),
    ('Random Forest', metrics_rf),
    ('MLP (PyTorch)', metrics_mlp)
]

class_labels = ['Non-Graduate', 'Graduate'] if 'graduate' == 'graduate' else ['Non-Graduate', 'Graduate']

for idx, (name, metrics_obj) in enumerate(models_info):
    # 직접 구현한 오차 행렬 호출
    cm = metrics_obj.confusion_matrix()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels,
                yticklabels=class_labels,
                ax=axes[idx])
    axes[idx].set_title(f'{name} (Custom CM)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices Comparison (Using Custom Metrics Class)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('[OK] Confusion matrices visualization complete')

In [ ]:
print('='*70)
print('KEY FINDINGS - TASK EVALUATION')
print('='*70)

best_model_name = comparison_df.loc[best_idx, 'Model']
best_f1 = comparison_df.loc[best_idx, 'F1-Score (Class 1)']

print(f'\n[1] Top Performing Model: {best_model_name}')
print(f'    - Accuracy: {comparison_df.loc[best_idx, "Accuracy"]:.4f}')
print(f'    - Target Class F1-Score: {best_f1:.4f}')
print(f'    - Best for interpretability and performance')

print('\n[2] ML vs DL Comparison')
ml_f1 = (f1s_pos[0] + f1s_pos[1] + f1s_pos[2]) / 3
dl_f1 = f1s_pos[3]
print(f'    - ML Average F1: {ml_f1:.4f}')
print(f'    - DL (PyTorch MLP) F1: {dl_f1:.4f}')
if ml_f1 > dl_f1:
    print(f'    - Machine learning models perform {(ml_f1 - dl_f1)/dl_f1*100:.1f}% better on this tabular dataset.')
else:
    print(f'    - Deep learning model performs {(dl_f1 - ml_f1)/ml_f1*100:.1f}% better on this tabular dataset.')

print('\n[3] Recommendation')
print(f'    - Deploy {best_model_name} model')
print('    - Reason: High target class prediction performance, fast inference, and robust learning.')

print('\n' + '='*70)